# Exercise: Rolling Forecast Backtest

Backtest ARIMA and SARIMAX on electricity demand using an expanding window. Compare cumulative MAE and bias.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from statsmodels.tsa.statespace.sarimax import SARIMAX


In [ ]:
import matplotlib as mpl

UB = {"Brand Blue": "#175CFF", "Medium Blue": "#7BA2FF"}
UN = {"Black": "#0A0B0F", "Dark Gray": "#5A5C62"}
mpl.rcParams["lines.linewidth"] = 3
mpl.rcParams["axes.linewidth"] = 2


In [ ]:
DATA_PATH = "../data/electricity_demand.csv"
HORIZON = 6


In [ ]:
def load_data(path):
    """
    Load electricity demand and return demand + temp series.
    """
    df = pd.read_csv(path, parse_dates=..., index_col=...).asfreq("MS")
    return df["demand_mwh"], df["avg_temp_f"]

demand, temp = load_data(DATA_PATH)
print(f"Months: {len(demand)}")


In [ ]:
def rolling_backtest(demand, temp, min_train=60, horizon=6):
    """
    Expanding-window backtest for ARIMA and SARIMAX.
    Returns dict of errors.
    """
    arima_errors = []
    sarimax_errors = []
    for i in range(min_train, len(demand) - horizon):
        train_d = demand.iloc[:i]
        train_t = temp.iloc[:i]
        actual = demand.iloc[i:i + horizon]

        arima = SARIMAX(train_d, order=...).fit(disp=False)
        arima_fc = arima.get_forecast(steps=...).predicted_mean
        arima_errors.extend(actual.values - arima_fc.values)

        sarima = SARIMAX(train_d, exog=train_t, order=..., seasonal_order=...).fit(disp=False)
        sarima_fc = sarima.get_forecast(steps=..., exog=temp.iloc[i:i + horizon]).predicted_mean
        sarimax_errors.extend(actual.values - sarima_fc.values)

    return {
        "arima_mae": np.mean(np.abs(arima_errors)),
        "arima_bias": np.mean(arima_errors),
        "sarimax_mae": np.mean(np.abs(sarimax_errors)),
        "sarimax_bias": np.mean(sarimax_errors)
    }

metrics = rolling_backtest(demand, temp)
for k, v in metrics.items():
    print(f"{k}: {v:,.0f}")


In [ ]:
def plot_error_comparison(metrics):
    """
    Bar chart comparing MAE and bias for both models.
    """
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    models = ["ARIMA", "SARIMAX"]
    mae_vals = [metrics["arima_mae"], metrics["sarimax_mae"]]
    bias_vals = [metrics["arima_bias"], metrics["sarimax_bias"]]
    axes[0].bar(models, mae_vals, color=[UB["Brand Blue"], UB["Medium Blue"]])
    axes[0].set_title("Mean Absolute Error", fontsize=18, fontweight="bold")
    axes[0].set_ylabel("MAE (MWh)", fontsize=16)
    axes[0].tick_params(axis="both", labelsize=14)
    axes[1].bar(models, bias_vals, color=[UB["Brand Blue"], UB["Medium Blue"]])
    axes[1].set_title("Mean Bias", fontsize=18, fontweight="bold")
    axes[1].set_ylabel("Bias (MWh)", fontsize=16)
    axes[1].tick_params(axis="both", labelsize=14)
    plt.tight_layout()
    plt.show()

plot_error_comparison(metrics)
